this is where we are going to pull rainfall for 2026 monsoon, without api keys


In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date, timedelta
from pathlib import Path
import osmnx as ox

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

LAT, LON = 26.8467, 80.9462
end = date.today() - timedelta(days=5)  # archive data lags ~5 days

r = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": LAT, "longitude": LON,
        "start_date": "2026-01-01", "end_date": end.isoformat(),
        "hourly": "precipitation,temperature_2m,relative_humidity_2m",
        "timezone": "Asia/Kolkata",
    },
    timeout=60,
)
r.raise_for_status()

rain = pd.DataFrame(r.json()["hourly"])
rain["time"] = pd.to_datetime(rain["time"])
rain.to_csv(RAW / "rainfall_data.csv", index=False)

print(rain.shape)
print("Total rain (mm):", round(rain["precipitation"].sum(), 1))
print("\nTop 10 heaviest hours:")
print(rain.nlargest(10, "precipitation")[["time", "precipitation"]])
daily = rain.set_index("time")["precipitation"].resample("D").agg(["sum", "max"])
daily.columns = ["total_mm", "peak_hour_mm"]
print(daily.sort_values("total_mm", ascending=False).head(15).round(1))

# Daily totals plot
daily = rain.set_index("time")["precipitation"].resample("D").agg(["sum", "max"])
daily.columns = ["total_mm", "peak_hour_mm"]
daily.plot(kind="bar", figsize=(14, 4), width=1.0)
plt.ylabel("mm/day"); plt.title("Lucknow daily rainfall, 2026")
plt.xticks([])
plt.tight_layout()
plt.savefig(RAW / "rain_2026.png")
plt.show()

In [ ]:
top = daily.sort_values("total_mm", ascending=False).head(15).round(1)
top.index = top.index.strftime("%d %b %Y")
print(top)

# Optional: a version of the plot with actual dates on the x-axis
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(daily.index, daily["total_mm"], width=1.0)
ax.set_ylabel("mm/day"); ax.set_title("Lucknow daily rainfall, 2026")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
plt.tight_layout(); plt.show()



In [ ]:
top = daily.sort_values("total_mm", ascending=False).head(15).round(1)
top.index = top.index.strftime("%d %b %Y")
print(top)

In [ ]:
places = ["Hazratganj", "Gomti Nagar", "Aliganj", "Charbagh", "Chowk", "Indira Nagar", "Alambagh"]
for p in places:
    try:
        lat, lon = ox.geocode(f"{p}, Lucknow, India")
        print(f"{p},{lat:.4f},{lon:.4f}")
    except Exception as e:
        print(p, "-> not found", e)

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LABELS = ROOT / "data" / "labels"
LABELS.mkdir(parents=True, exist_ok=True)

url = "https://www.newsdrum.in/national/late-september-deluge-batters-north-india-up-schools-shut-uttarakhand-on-red-alert-12577325"
places = {
    "Hazratganj":   (26.8548, 80.9448),
    "Gomti Nagar":  (26.8612, 81.0039),
    "Aliganj":      (26.9051, 80.9480),
    "Charbagh":     (26.8335, 80.9264),
    "Chowk":        (26.8677, 80.9042),
    "Indira Nagar": (26.8823, 80.9900),
    "Alambagh":     (26.8140, 80.9025),
}

rows = [
    {"date": "2026-09-25", "location_name": name, "lat": lat, "lon": lon,
     "severity": "high", "source": "NewsDrum", "source_url": url}
    for name, (lat, lon) in places.items()
]

df = pd.DataFrame(rows)
df.to_csv(LABELS / "waterlogging_reports.csv", index=False)
print(df)

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

end = (date.today() - timedelta(days=5)).isoformat()
r = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": 26.8467, "longitude": 80.9462,
        "start_date": "2022-01-01", "end_date": end,
        "hourly": "precipitation,temperature_2m,relative_humidity_2m",
        "timezone": "Asia/Kolkata",
    },
    timeout=120,
)
r.raise_for_status()
rain_all = pd.DataFrame(r.json()["hourly"])
rain_all["time"] = pd.to_datetime(rain_all["time"])
rain_all.to_csv(RAW / "rainfall_2022_2026.csv", index=False)
print(rain_all.shape)

daily = rain_all.set_index("time")["precipitation"].resample("D").agg(["sum", "max"])
daily.columns = ["total_mm", "peak_hour_mm"]
daily.index = daily.index.strftime("%d %b %Y")
print(daily.sort_values("total_mm", ascending=False).head(25).round(1))